# Catboost


## everything but catboost now

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import joblib

from src.config import SEED, MODEL_DIR
from src.data import load_train, get_split
from src.preprocessing import learn_preprocessing, apply_preprocessing
from src.models import make_catboost
from src.evaluate import report

In [3]:
X, y = load_train()
X_tr, X_val, y_tr, y_val = get_split(X, y)
print(X_tr.shape, X_val.shape)

(559708, 21) (139927, 21)


In [4]:
state_num = learn_preprocessing(X_tr, ratings="numeric", scaling="no")

X_tr_num = apply_preprocessing(X_tr, state_num)
X_val_num = apply_preprocessing(X_val, state_num)

In [5]:
model = make_catboost()
model.fit(X_tr_num, y_tr)

CatBoostClassifier(depth=6, iterations=500, learning_rate=0.05, random_seed=42, verbose=False)

In [6]:
train_auc = roc_auc_score(
    y_tr, model.predict_proba(X_tr_num)[:, 1]
)
print(f"Train AUC: {train_auc:.4f}")

Train AUC: 0.9592


In [7]:
auc = report(model, X_val_num, y_val)

Validation ROC-AUC : 0.9566
Validation accuracy: 0.9237

Confusion matrix:
 [[73276  4583]
 [ 6093 55975]] 

              precision    recall  f1-score   support

           0       0.92      0.94      0.93     77859
           1       0.92      0.90      0.91     62068

    accuracy                           0.92    139927
   macro avg       0.92      0.92      0.92    139927
weighted avg       0.92      0.92      0.92    139927



In [8]:
from src.data import load_test
from src.submission import save_submission

In [10]:
test_ids, X_test = load_test()
X_test_encoded = apply_preprocessing(X_test, state_num)
test_proba = model.predict_proba(X_test_encoded)[:, 1]

In [11]:
save_submission(test_ids, test_proba, "catboost.csv")

Saved to: C:\Users\Bharat\Desktop\remaining-2026\Career-Tracks\Track 01 - Data Science\Relevant Projects\airline-satisfaction-prediction\submissions\catboost.csv
count    299844.000000
mean          0.443558
std           0.426375
min           0.002099
25%           0.036876
50%           0.201082
75%           0.947637
max           0.996499
Name: satisfaction, dtype: float64


,id,satisfaction
0,699635,0.242809
1,699636,0.903667
2,699637,0.867649
3,699638,0.049817
4,699639,0.054514
...,...,...
299839,999474,0.067518
299840,999475,0.057024
299841,999476,0.290224
299842,999477,0.031897
